#1.모델 선정
실시간 검색 -> 퍼플렉시티

추론 및 창작 -> Claude

다용도 활용 -> ChatGPT

일상 및 간단 -> Llama

데이터 분석 -> Gemini

#2 모델 배정
5개 모델에게 최적의 모델 질문 -> 다수결 투표로 모델 배정

#3 최종 답변
프롬프트 엔지니어링 적용 후 질문을 통해 답변 생성



In [ ]:
!pip install -q -U langchain-openai streamlit

In [ ]:
import os
# 환경 설정 및 API 키 입력 (코랩 보안 비밀 전용 코드)
from google.colab import userdata

openrouter_key = userdata.get('OPENROUTER_API_KEY')

with open("app.py", "w") as f:
    f.write(f'''
import streamlit as st
import os
import json
import base64
import concurrent.futures
from collections import Counter
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

# 대화 기록 기능
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# 1. API 키 설정 (오픈라우터 전용)
os.environ["OPENROUTER_API_KEY"] = "{openrouter_key}"

# 대화 기록 직렬화
def convert_chat_to_json():
    if "chat_history" in st.session_state and st.session_state.chat_history:
        serializable_history = []
        for msg in st.session_state.chat_history:
            content_str = ""
            if isinstance(msg.content, list):
                for item in msg.content:
                    if isinstance(item, dict) and item.get("type") == "text":
                        content_str += item.get("text", "")
            else:
                content_str = str(msg.content)

            if isinstance(msg, HumanMessage):
                serializable_history.append({{"role": "user", "content": content_str}})
            elif isinstance(msg, AIMessage):
                serializable_history.append({{"role": "assistant", "content": content_str}})

        return json.dumps(serializable_history, ensure_ascii=False, indent=2)
    return ""

# 2. 데이터 구조 정의
class RouteResponse(BaseModel):
    selected_model: str = Field(description="선택된 최적의 모델 (Llama, Gemini, ChatGPT, Claude, Perplexity 중 택 1)")
    model_reason: str = Field(description="5개 모델의 앙상블 판단 결과, 해당 모델을 선택한 명확한 이유")

class OptimizeResponse(BaseModel):
    optimized_prompt: str = Field(description="맥락이 복원된 완벽한 하나의 문장")

# 3. OpenRouter 기반 AI 모델 초기화
def get_openrouter_llm(model_id, temperature=0.7):
    return ChatOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.environ["OPENROUTER_API_KEY"],
        model=model_id,
        temperature=temperature
    )

# [맥락 복원 전담 모델] 빠르고 저렴한 모델로 고정 (프롬프트 최적화용)
context_optimizer = get_openrouter_llm("openai/gpt-4o-mini", temperature=0)

# [답변 생성용 5대 전문 모델]
models_dict = {{
    "Llama": get_openrouter_llm("meta-llama/llama-3.3-70b-instruct"),
    "Gemini": get_openrouter_llm("google/gemini-2.5-pro"),
    "ChatGPT": get_openrouter_llm("openai/gpt-4o"),
    "Claude": get_openrouter_llm("anthropic/claude-sonnet-4.6"),
    "Perplexity": get_openrouter_llm("perplexity/sonar-pro")
}}

# 4. [1단계] 맥락 복원
def optimize_prompt(user_input: str, chat_history: list):
    if not chat_history:
        return user_input # 첫 질문이면 그대로 통과

    history_lines = []
    for m in chat_history[-6:]:
        role = "User" if isinstance(m, HumanMessage) else "AI"
        history_lines.append(f"{{role}}: {{m.content}}")

    history_text = "\\n".join(history_lines)
    # 대화 기록 속 코드 중괄호 무효화 (Escape)
    history_text = history_text.replace("{{", "{{{{").replace("}}", "}}}}")

    system_prompt = f"""당신은 대화 맥락을 파악하여 사용자의 짧은 질문을 '독립적으로 이해 가능한 완벽한 하나의 문장'으로 재구성하는 프롬프트 엔지니어입니다.
    대명사(그거, 걔, 저번에 등)나 생략된 주어를 아래 [이전 대화 맥락]을 참고해 구체적인 명사로 치환하세요.
    질문이 이미 명확하다면 원래 입력을 그대로 유지하세요.

    [이전 대화 맥락]
    {{history_text}}

    반드시 JSON 형식 {{{{{{{{ "optimized_prompt": "..." }}}}}}}} 으로만 출력하세요."""

    try:
        prompt = ChatPromptTemplate.from_messages([("system", system_prompt), ("human", "{{input}}")])
        chain = prompt | context_optimizer | JsonOutputParser(pydantic_object=OptimizeResponse)
        opt_result = chain.invoke({{"input": user_input}})
        return opt_result.get("optimized_prompt", user_input)
    except Exception:
        return user_input

# 5. [2단계] 다수결 앙상블 라우터
def get_single_vote(model_name, model_instance, system_prompt, user_input):
    prompt = ChatPromptTemplate.from_messages([("system", system_prompt), ("human", "{{input}}")])
    chain = prompt | model_instance | JsonOutputParser(pydantic_object=RouteResponse)
    try:
        result = chain.invoke({{"input": user_input}})
        selected = result.get("selected_model", "Gemini")
        if selected not in ["Llama", "Gemini", "ChatGPT", "Claude", "Perplexity"]:
            selected = "Gemini"
        return {{"voter": model_name, "selected_model": selected, "reason": result.get("model_reason", "")}}
    except Exception as e:
        return {{"voter": model_name, "selected_model": "Gemini", "reason": f"에러: {{e}}"}}

def route_to_model(optimized_prompt, chat_history):
    history_lines = []
    for m in chat_history[-4:]:
        role = "User" if isinstance(m, HumanMessage) else "AI"
        history_lines.append(f"{{role}}: {{m.content}}")

    history_text = "\\n".join(history_lines)
    # 대화 기록 속 코드 중괄호 무효화 (Escape)
    history_text = history_text.replace("{{", "{{{{").replace("}}", "}}}}")

    system_prompt = f"""당신은 5개의 AI 모델 중 하나입니다. 사용자의 질문을 분석하여, 어떤 모델이 답변하는 것이 가장 완벽할지 객관적으로 투표하세요.

    [배정 가능 모델 가이드라인 - 필독]
    1. Perplexity (특수/검색): 2025년/2026년 최신 정보, 뉴스, 현재 날씨, 최근 스포츠 결과 등 '실시간 인터넷 검색'이 반드시 필요한 질문.
    2. Claude (헤비급/추론): 매우 복잡한 코딩(웹 아키텍처 등), 알고리즘 설계, 깊은 수학/논리적 추론, 고품질 글쓰기.
    3. ChatGPT (헤비급/범용): GPT-4o 수준의 다단계 논리 해결, 엑셀/데이터 구조화, 일반적인 어려운 문제 해결.
    4. Llama (헤비급/CS): C, Java, Python 등 프로그래밍 기본기, 메모리 할당, 포인터, 비트 OR 연산(|) 등 컴퓨터 공학 개념 설명.
    5. Gemini (헤비급/데이터): 방대한 텍스트 처리, 복잡한 스포츠 승률 및 확률 데이터 분석, 긴 문서 요약 및 교차 검증.

    [이전 대화 맥락]
    {{history_text}}

    반드시 아래 JSON 형식으로만 답변하세요:
    {{{{{{{{
        "selected_model": "Llama, Gemini, ChatGPT, Claude, Perplexity 중 하나",
        "model_reason": "위 가이드라인에 근거한 구체적인 투표 사유"
    }}}}}}}}"""

    votes = []
    with concurrent.futures.ThreadPoolExecutor() as executor:
        futures = [
            executor.submit(get_single_vote, name, inst, system_prompt, optimized_prompt)
            for name, inst in models_dict.items()
        ]
        for future in concurrent.futures.as_completed(futures):
            votes.append(future.result())

    voted_models = [v["selected_model"] for v in votes]
    vote_counts = Counter(voted_models)

    max_votes = max(vote_counts.values())
    top_models = [model for model, count in vote_counts.items() if count == max_votes]

    if len(top_models) > 1:
        chatgpt_vote = next((v["selected_model"] for v in votes if v["voter"] == "ChatGPT"), "ChatGPT")
        final_model_name = chatgpt_vote if chatgpt_vote in top_models else top_models[0]
        tie_break_msg = f"(동률 발생! ChatGPT의 투표 반영: {{final_model_name}})"
    else:
        final_model_name = top_models[0]
        tie_break_msg = f"(다수결 단독 1위: {{final_model_name}})"

    final_reason = next(
        (v["reason"] for v in votes if v["selected_model"] == final_model_name and v["voter"] == final_model_name and "에러" not in v["reason"]),
        next((v["reason"] for v in votes if v["selected_model"] == final_model_name and "에러" not in v["reason"]), "다수결에 의한 선택")
    )

    return {{
        "model_name": final_model_name,
        "model_reason": f"{{vote_counts}} {{tie_break_msg}}\\n최종 사유: {{final_reason}}",
        "model_instance": models_dict[final_model_name],
        "optimized_prompt": optimized_prompt
    }}

# 6. [3단계] 최종 답변 생성 함수
def generate_final_answer(optimized_result: dict, chat_history: list, file_context: dict = None, is_stream: bool = True):
    if not optimized_result: return None

    prompt = optimized_result.get('optimized_prompt', '')
    model = optimized_result.get('model_instance')

    try:
        messages = [
            SystemMessage(content="""당신은 친절한 AI 어시스턴트입니다.
            [매우 중요한 지침 - 반드시 지킬 것]
            1. 이전 대화 기록은 다른 특화 전문가 AI들이 작성한 '절대적인 팩트'입니다. 이전 대화의 사실 여부를 스스로 판단하여 지적하거나, 정정하거나, 사과하지 마세요.
            2. 오직 사용자가 가장 마지막에 입력한 '현재 질문'에 대해서만 집중해서 답변을 생성하세요.
            3. 사용자가 제공한 첨부 파일(문서 텍스트 또는 이미지)의 내용을 최우선으로 인식하여 답변에 반영하세요.
            4. 기본적으로 한국어로 답변하되, 사용자가 특정 언어로 작성이나 번역을 요청할 경우 반드시 해당 언어를 사용하세요.""")
        ]
        messages.extend(chat_history)
        messages.append(HumanMessage(content=prompt))

        # 파일 컨텍스트가 존재할 경우 유저 메시지를 고도화
        if file_context:
            user_content = []

            # 텍스트 파일 내용 주입
            if file_context.get("text_content"):
                user_content.append({{"type": "text", "text": f"[참조 첨부 문서 내용]\\n{{file_context['text_content']}}\\n\\n"}} )

            # 이미지 파일 데이터 주입 (Base64 멀티모달 전송 표준 규격)
            if file_context.get("image_base64"):
                user_content.append({{
                    "type": "image_url",
                    "image_url": {{"url": f"data:image/jpeg;base64,{{file_context['image_base64']}}"}}
                }})

            # 최종 질문 주입
            user_content.append({{"type": "text", "text": f"질문: {{prompt}}"}} )
            messages.append(HumanMessage(content=user_content))
        else:
            messages.append(HumanMessage(content=prompt))

        # 사용자가 토글을 켰을 때: 스트리밍 제너레이터 함수 반환
        if is_stream:
            def stream_generator():
                for chunk in model.stream(messages):
                    content = chunk.content if hasattr(chunk, 'content') else str(chunk)
                    yield content
            return {{**optimized_result, "stream_object": stream_generator(), "is_stream_mode": True}}

        # 사용자가 토글을 껐을 때: 기존 동기식 통째로 연산(invoke)
        else:
            response = model.invoke(messages)
            answer_text = response.content if hasattr(response, 'content') else str(response)
            return {{**optimized_result, "answer": answer_text, "is_stream_mode": False}}

    except Exception as e:
        return {{**optimized_result, "answer": f"답변 생성 중 오류가 발생했습니다: {{e}}", "is_stream_mode": False}}

# 7. Streamlit UI 로직
st.set_page_config(page_title="VIKI: 프롬프트 최적화 기반 지능형 AI 응답 시스템", layout="wide")

# 세션 구조 정의
if "chat_history" not in st.session_state:
    st.session_state.chat_history = []

file_payload = None

# 사이드바
with st.sidebar:
    st.title("⚙️ VIKI 제어 시스템")

    st.subheader("💡 사용자 설정")
    stream_mode = st.toggle("실시간 답변 스트리밍 활성화", value=True)

    st.divider()

    # 대화 기록
    st.subheader("💾 데이터 관리")
    if st.button("🧹 대화 기록 초기화", type="primary", use_container_width=True):
        st.session_state.chat_history = []
        st.rerun()

    st.write("")

    chat_json_data = convert_chat_to_json()
    if chat_json_data:
        st.download_button(
            label="📥 현재 대화 기록 다운로드",
            data=chat_json_data,
            file_name="viki_chat_history.json",
            mime="application/json",
            use_container_width=True
        )
    else:
        st.button("📥 현재 대화 기록 다운로드", disabled=True, use_container_width=True)
        st.caption("ℹ️ 대화 기록이 쌓이면 다운로드 버튼이 활성화됩니다.")

    st.divider()

    st.subheader("📁 파일 업로드")
    # 드래그 앤 드롭 파일 업로더 생성
    uploaded_file = st.file_uploader(
        "파일 업로드",
        type=["png", "jpg", "jpeg", "txt", "pdf", "docx", "xlsx", "csv", "ppt", "pptx", "py", "ipynb"],
        help="지원 형식: PNG, JPG, JPEG, TXT, PDF, DOCX, XLSX, CSV, PPT, PPTX, PY, IPYNB"
    )

    if uploaded_file:
        st.success(f"{{uploaded_file.name}} 정상 연동됨")
        file_payload = {{"text_content": None, "image_base64": None}}

        if uploaded_file.type.startswith("image"):
            st.image(uploaded_file, caption="업로드 이미지 미리보기", use_container_width=True)
            bytes_data = uploaded_file.getvalue()
            file_payload["image_base64"] = base64.b64encode(bytes_data).decode("utf-8")

        elif uploaded_file.name.endswith(".py") or uploaded_file.name.endswith(".ipynb") or uploaded_file.type == "text/plain":
            bytes_data = uploaded_file.getvalue()
            raw_text = bytes_data.decode("utf-8", errors="ignore")

            if uploaded_file.name.endswith(".ipynb"):
                try:
                    notebook_obj = json.loads(raw_text)
                    code_lines = []
                    for cell in notebook_obj.get("cells", []):
                        if cell.get("cell_type") == "code":
                            code_lines.append("".join(cell.get("source", [])))
                    file_payload["text_content"] = "\\n# --- New Code Cell --- \\n".join(code_lines)
                except Exception:
                    file_payload["text_content"] = raw_text
            else:
                file_payload["text_content"] = raw_text
        else:
            file_payload["text_content"] = f"[첨부파일명: {{uploaded_file.name}} 문서가 업로드되었습니다.]"

st.title("🤖 VIKI: 프롬프트 최적화 기반 지능형 AI 응답 시스템")

# 대화 기록이 없을 때만 나타나는 환영 메시지
if not st.session_state.chat_history:
    st.info("👋 안녕하세요! VIKI입니다. 최신 정보 검색부터 복잡한 코드 설계까지 당신의 질문에 최적화된 답변을 제공합니다.")

for msg in st.session_state.chat_history:
    display_content = ""
    if isinstance(msg.content, list):
        for item in msg.content:
            if isinstance(item, dict) and item.get("type") == "text":
                display_content += item.get("text", "")
    else:
        display_content = str(msg.content)

    with st.chat_message("user" if isinstance(msg, HumanMessage) else "assistant"):
        st.write(display_content)

if query := st.chat_input("질문을 입력하세요..."):
    with st.chat_message("user"):
        st.write(query)

    with st.spinner("VIKI 엔진 가동 중..."):
        step1_optimized = optimize_prompt(query, st.session_state.chat_history)
        step2_routed = route_to_model(step1_optimized, st.session_state.chat_history)
        final = generate_final_answer(step2_routed, st.session_state.chat_history, file_context=file_payload, is_stream=stream_mode)

        if final:
            with st.expander("🔍 VIKI 시스템 추론 로그"):
                st.write(f"**원본 질문:** {{query}}")
                st.write(f"**맥락 복원 질문:** {{final.get('optimized_prompt')}}")
                st.write(f"**배정 모델:** {{final.get('model_name')}}")
                st.write(f"**배정 사유:** {{final.get('model_reason')}}")

            with st.chat_message("assistant"):
                # 케이스 A: 스트리밍 활성화 모드인 경우 -> 실시간 제너레이터 출력
                if final.get("is_stream_mode"):
                    answer = st.write_stream(final.get("stream_object"))
                # 케이스 B: 스트리밍 비활성화 모드인 경우 -> 텍스트 한 번에 출력
                else:
                    answer = final['answer']
                    st.write(answer)

            st.session_state.chat_history.append(HumanMessage(content=query))
            st.session_state.chat_history.append(AIMessage(content=answer))
''')

In [ ]:
!npm install -g localtunnel

# 1. 보안 비밀번호(IP) 확인 (접속 시 필요함)
import urllib
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n")
print(f"Tunnel Password (IP): {ip}")

# 2. Streamlit 서버 실행 및 외부 링크 생성
!streamlit run app.py & npx localtunnel --port 8501